 # Walmart Sales Data Analysis 
**Author:** Christos Bouzios  
**Objective:** To clean raw retail data using Pandas, load it into a local SQLite database, and extract business insights using advanced SQL querying (Window Functions, CTEs).
***

In [117]:
import pandas as pd
from sqlalchemy import create_engine

### 1. Data Loading & Initial Exploration
First, we load the raw dataset and inspect its structure, dimensions, and data types.

In [118]:
df = pd.read_csv("data_raw/Walmart.csv", encoding_errors='ignore')

df.shape

(10051, 11)

In [119]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


### 2. Data Cleaning & Feature Engineering
The raw data requires formatting before it can be reliably queried in SQL. In this section, we:
* Standardize column names
* Remove exact duplicate rows and handle missing values
* Clean the `unit_price` formatting and convert it to a float
* Engineer a new `total` revenue column

In [120]:
## ΜΕΤΑΜΟΡΦΩΝΩ ΟΛΕΣ ΤΙΣ ΣΤΗΛΕΣ ΣΕ lower case
df.columns = df.columns.str.lower()

In [121]:
## ΟΙ ΤΙΜΕΣ ΣΤΟ count ΔΙΑΦΕΡΟΥΝ (10051 != 10020)
df.describe()

,invoice_id,quantity,rating,profit_margin
count,10051.000000,10020.000000,10051.000000,10051.000000
mean,5025.741220,2.353493,5.825659,0.393791
std,2901.174372,1.602658,1.763991,0.090669
min,1.000000,1.000000,3.000000,0.180000
25%,2513.500000,1.000000,4.000000,0.330000
50%,5026.000000,2.000000,6.000000,0.330000
75%,7538.500000,3.000000,7.000000,0.480000
max,10000.000000,10.000000,10.000000,0.570000


In [122]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   branch          10051 non-null  str    
 2   city            10051 non-null  str    
 3   category        10051 non-null  str    
 4   unit_price      10020 non-null  str    
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  str    
 7   time            10051 non-null  str    
 8   payment_method  10051 non-null  str    
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), str(7)
memory usage: 863.9 KB


In [123]:
## ΔΙΠΛΟΤΥΠΑ = 51
df.duplicated().sum()

np.int64(51)

In [124]:
df.isnull().sum()

invoice_id         0
branch             0
city               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method     0
rating             0
profit_margin      0
dtype: int64

In [125]:
## ΑΦΑΙΡΩ ΜΗΔΕΝΙΚΕΣ ΤΙΜΕΣ ΑΠΟ ΤΟ dataframe ΚΑΙ ΤΑ ΔΙΠΛΟΤΥΠΑ
df.dropna(inplace = True)
df.drop_duplicates()

##ΕΠΙΒΕΒΑΙΩΝΩ
df.isnull().sum()

invoice_id        0
branch            0
city              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

In [126]:
df.shape

(10020, 11)

In [127]:
## ΜΕΤΑΤΡΕΠΩ ΤΗ ΣΤΗΛΗ 'UNIT_PRICE' ΑΠΟ object ΣΕ float ΓΙΑ ΝΑ ΜΠΟΡΩ ΝΑ ΚΑΝΩ ΠΡΑΞΕΙΣ
df['unit_price'] = df['unit_price'].str.replace('$','').astype(float)

In [128]:
df.info()

<class 'pandas.DataFrame'>
Index: 10020 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10020 non-null  int64  
 1   branch          10020 non-null  str    
 2   city            10020 non-null  str    
 3   category        10020 non-null  str    
 4   unit_price      10020 non-null  float64
 5   quantity        10020 non-null  float64
 6   date            10020 non-null  str    
 7   time            10020 non-null  str    
 8   payment_method  10020 non-null  str    
 9   rating          10020 non-null  float64
 10  profit_margin   10020 non-null  float64
dtypes: float64(4), int64(1), str(6)
memory usage: 939.4 KB


In [129]:
## ΔΗΜΙΟΥΡΓΩ ΝΕΑ ΣΤΗΛΗ ΜΕ ΤΟ ΣΥΝΟΛΟ ΠΟΥ ΠΛΗΡΩΘΗΚΕ
df['total'] = df['unit_price'] * df['quantity']

In [130]:
df.shape

(10020, 12)

### 3. Database Integration (SQLite)
With the data cleaned, we establish a local SQLite database using SQLAlchemy and export our Pandas DataFrame directly into a new SQL table named `sales_data`.

In [131]:
## ΣΥΝΔΕΣΗ ΜΕ ΤΟΠΙΚΟ ΑΡΧΕΙΟ 
engine = create_engine("sqlite:///my_project.db")

## ΠΙΝΑΚΑΣ sales_data ΟΠΟΥ ΜΕΤΑΦΕΡΩ ΤΑ ΔΕΔΟΜΕΝΑ ΤΟΥ dataframe ΣΕ sql
df.to_sql('sales_data', con=engine, if_exists='replace', index=False)

print("ΤΑ ΔΕΔΟΜΕΝΑ ΜΕΤΑΦΕΡΘΗΚΑΝ ΣΕ SQL")

ΤΑ ΔΕΔΟΜΕΝΑ ΜΕΤΑΦΕΡΘΗΚΑΝ ΣΕ SQL


### 4. Advanced SQL Analysis
Now we transition from Python to SQL. We will use the `pd.read_sql()` function to execute complex queries against our SQLite database to answer critical business questions.

In [132]:
query = """
SELECT *
FROM sales_data
LIMIT 10
"""

## Τρέχουμε το query μέσα από τη σύνδεση (engine) και τυπώνουμε το αποτέλεσμα
query_results= pd.read_sql(query, con=engine)
query_results

,invoice_id,branch,city,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48,76.40
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33,324.31
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33,465.76
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48,604.17
5,6,WALM026,Denton,Electronic accessories,85.39,7.0,25/03/19,18:30:00,Ewallet,4.1,0.48,597.73
6,7,WALM088,Cleburne,Electronic accessories,68.84,6.0,25/02/19,14:36:00,Ewallet,5.8,0.33,413.04
7,8,WALM100,Canyon,Home and lifestyle,73.56,10.0,24/02/19,11:38:00,Ewallet,8.0,0.18,735.60
8,9,WALM066,Grapevine,Health and beauty,36.26,2.0,10/01/19,17:15:00,Credit card,7.2,0.33,72.52
9,10,WALM065,Texas City,Food and beverages,54.84,3.0,20/02/19,13:27:00,Credit card,5.9,0.33,164.52


In [133]:
query = """
SELECT COUNT(*)
FROM sales_data
LIMIT 10
"""
query_results= pd.read_sql(query, con=engine)
query_results

,COUNT(*)
0,10020


In [ ]:
## ΒΡΕΣ ΓΙΑ ΚΑΘΕ ΔΙΑΦΟΡΕΙΤΚΗ ΜΕΘΟΔΟ ΠΛΗΡΩΜΗΣ, ΤΟΝ ΑΡΙΘΜΟ ΣΥΝΑΛΛΑΓΩΝ ΚΑΙ ΤΗΝ ΠΟΣΟΤΗΤΑ ΠΩΛΗΘΕΝΤΩΝ

query = """
SELECT 
    payment_method,
    COUNT(*) as no_of_payments,
    SUM(quantity) as no_qty_sold
 FROM sales_data
 GROUP BY payment_method
"""
query_results= pd.read_sql(query, con=engine)
query_results

,payment_method,no_of_payments,no_qty_sold
0,Cash,1880,5077.0
1,Credit card,4259,9573.0
2,Ewallet,3881,8932.0


In [ ]:
## ΠΡΟΣΔΙΟΡΙΣΕ ΤΗΝ ΚΑΤΗΓΟΡΙΑ ΜΕ ΤΗΝ ΥΨΗΛΟΤΕΡΗ ΒΑΘΜΟΛΟΓΙΑ ΣΕ ΚΑΘΕ ΥΠΟΚΑΤΑΣΤΗΜΑ, ΕΜΦΑΝΙΖΟΝΤΑΣ ΤΟ ΥΠΟΚΑΤΑΣΤΗΜΑ,
## ΤΗΝ ΚΑΤΗΓΟΡΙΑ ΚΑΙ ΤΗ ΜΕΣΗ ΒΑΘΜΟΛΟΓΙΑ

query = """
SELECT *
FROM 
    (SELECT 
        branch,
        category,
        AVG(rating) as avg_rating,
        RANK() OVER(PARTITION BY branch ORDER BY AVG(rating) DESC) as rank
        FROM sales_data
    GROUP BY branch, category)
WHERE rank = 1
"""
query_results= pd.read_sql(query, con=engine)
query_results

,branch,category,avg_rating,rank
0,WALM001,Electronic accessories,7.450000,1
1,WALM002,Food and beverages,8.250000,1
2,WALM003,Sports and travel,7.500000,1
3,WALM004,Food and beverages,9.300000,1
4,WALM005,Health and beauty,8.366667,1
...,...,...,...,...
96,WALM096,Sports and travel,9.600000,1
97,WALM097,Food and beverages,7.675000,1
98,WALM098,Health and beauty,9.800000,1
99,WALM099,Electronic accessories,5.950000,1


In [147]:
## ΠΡΟΣΔΙΟΡΙΣΕ ΤΗΝ ΕΛΑΧΙΣΤΗ, ΜΕΓΙΣΤΗ ΚΑΙ ΜΕΣΣΗ ΒΑΘΜΟΛΟΓΙΑ ΚΑΤΗΓΟΡΙΑΣ ΓΙΑ ΚΑΘΕ ΠΟΛΗ

query = """
SELECT 
    city,
    category,
    MIN(rating) as min_rating,
    MAX(rating) as max_rating,
    AVG(rating) as avg_rating
FROM sales_data
GROUP BY city, category
"""
query_results= pd.read_sql(query, con=engine)
query_results

,city,category,min_rating,max_rating,avg_rating
0,Abilene,Electronic accessories,7.1,8.8,7.966667
1,Abilene,Fashion accessories,4.0,9.0,6.240625
2,Abilene,Food and beverages,6.0,8.9,6.950000
3,Abilene,Health and beauty,9.7,9.7,9.700000
4,Abilene,Home and lifestyle,4.0,9.0,6.096875
...,...,...,...,...,...
508,Weslaco,Fashion accessories,3.0,9.5,5.042012
509,Weslaco,Food and beverages,7.1,9.8,8.733333
510,Weslaco,Health and beauty,4.3,9.2,6.750000
511,Weslaco,Home and lifestyle,3.0,9.2,5.180711


In [ ]:
##ΕΜΦΑΝΙΣΕ ΤΟ ΚΑΤΑΣΤΗΜΑ ΚΑΙ ΤΗΝ ΠΡΟΤΙΜΟΤΕΡΗ ΜΕΘΟΔΟ ΠΛΗΡΩΜΗΣ

query = """
WITH cte
AS (
    SELECT 
        branch,
        payment_method,
        COUNT(*) as total_transactions,
        RANK() OVER(PARTITION BY branch ORDER BY COUNT(*) DESC) as rank
    FROM sales_data
GROUP BY branch, payment_method)

SELECT * 
FROM cte
WHERE rank = 1
"""
query_results= pd.read_sql(query, con=engine)
query_results

,branch,payment_method,total_transactions,rank
0,WALM001,Ewallet,45,1
1,WALM002,Ewallet,37,1
2,WALM003,Credit card,115,1
3,WALM004,Ewallet,44,1
4,WALM005,Ewallet,56,1
...,...,...,...,...
95,WALM096,Ewallet,50,1
96,WALM097,Ewallet,38,1
97,WALM098,Ewallet,44,1
98,WALM099,Credit card,83,1
